In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

25/11/28 15:24:01 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/11/28 15:24:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-75a9a717-b6ad-44e7-81fd-0a1298fb9545;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [2]:
# Cell 0 – Utils import (shared pattern)

import sys
import os
import importlib

# Path to the folder that contains utils_silver.py
# (adjust if your repo is in a different location)
project_root = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"
silver_dir = os.path.join(project_root, "_02_Silver")

if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

import utils_silver
importlib.reload(utils_silver)  # pick up edits during dev
from utils_silver import *

print("Imported utils_silver from:", utils_silver.__file__)


✅ utils_silver.py loaded
✅ utils_silver.py loaded
Imported utils_silver from: /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# Cell 1 — Setup, imports, paths

In [ ]:
# Cell 1 — Setup

from pyspark.sql import functions as F
from pyspark.sql.types import *
from utils_silver import (
    build_paths,
    enforce_schema,
    drop_dupes_keep_latest,
    fix_dates,
    dq_expect,
    dq_left_anti_ref,
    quarantine,
    write_metric
)

STORAGE_ACCOUNT = "clientdatastorage"

paths_bronze, paths_silver, paths_gold = build_paths(STORAGE_ACCOUNT)

BRONZE_CLAIMS_PATH = paths_bronze["claims"]
BRONZE_MEMBERS_PATH = paths_bronze["members"]
BRONZE_PROVIDERS_PATH = paths_bronze["providers"]

SILVER_CLAIMS_PATH = paths_silver["claims"]

print("Bronze claims:", BRONZE_CLAIMS_PATH)
print("Silver claims:", SILVER_CLAIMS_PATH)


Bronze claims: abfss://rawdata@clientdatastorage.dfs.core.windows.net/claims
Silver claims: abfss://silverdata@clientdatastorage.dfs.core.windows.net/claims


# Cell 2 — Read Bronze claims & enforce schema

In [4]:
# Cell 2 — Read Bronze

claims_bz = spark.read.format("delta").load(BRONZE_CLAIMS_PATH)
members_bz = spark.read.format("delta").load(BRONZE_MEMBERS_PATH)
providers_bz = spark.read.format("delta").load(BRONZE_PROVIDERS_PATH)

print("Bronze claims count:", claims_bz.count())
claims_bz.show(5, truncate=False)
claims_bz.printSchema()

# Enforce consistent schema
claims_schema = StructType([
    StructField("Claim_ID", StringType()),
    StructField("Provider_ID", StringType()),
    StructField("Member_Key", StringType()),
    StructField("Date_Reported", DateType()),
    StructField("Date_Settled", DateType()),
    StructField("Payout_GBP", DoubleType()),
    StructField("Claim_Amount_GBP", DoubleType()),
    StructField("Fraud_Label", IntegerType()),
    StructField("Claim_Type", StringType()),
    StructField("Claim_Status", StringType())
])

claims = enforce_schema(claims_bz, claims_schema)


25/11/28 15:24:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze claims count: 558211


+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|Claim_ID|Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|
+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|CLM46614|PRV55912   |BENE11001 |NULL         |NULL        |26000.0   |30397.346951642197|0          |Hospital  |Settled     |
|CLM66048|PRV55907   |BENE11001 |NULL         |NULL        |5000.0    |6379.96457105169  |0          |Hospital  |Settled     |
|CLM68358|PRV56046   |BENE11001 |NULL         |NULL        |5000.0    |7307.291499809586 |0          |Hospital  |Settled     |
|CLM38412|PRV52405   |BENE11011 |NULL         |NULL        |5000.0    |6989.424384581965 |0          |Hospital  |Settled     |
|CLM63689|PRV56614   |BENE11014 |NULL         |NULL        |10000.0   |13123.302569965643|0          |Hospital 

# ✅ Cell 3 — Fix dates + monetary sanity

In [5]:
# Cell 3 — Fix dates and monetary checks

claims = fix_dates(claims, "Date_Reported", "Date_Settled")

# Basic monetary logic
claims = (
    claims
    .withColumn("Payout_GBP", F.col("Payout_GBP").cast("double"))
    .withColumn("Claim_Amount_GBP", F.col("Claim_Amount_GBP").cast("double"))
)

claims.printSchema()
claims.show(5, truncate=False)


root
 |-- Claim_ID: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Date_Reported: date (nullable = true)
 |-- Date_Settled: date (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Fraud_Label: integer (nullable = true)
 |-- Claim_Type: string (nullable = true)
 |-- Claim_Status: string (nullable = true)



+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|CLM238855|PRV57674   |BENE50241 |NULL         |NULL        |200.0     |243.85551149793676|0          |Dental    |Settled     |
|CLM312840|PRV57674   |BENE50241 |NULL         |NULL        |60.0      |93.52525212507253 |0          |Hospital  |Rejected    |
|CLM708959|PRV57674   |BENE50241 |NULL         |NULL        |300.0     |426.6228985021468 |0          |Hospital  |Settled     |
|CLM147537|PRV52066   |BENE50242 |NULL         |NULL        |90.0      |98.83400485730229 |0          |Hospital  |Settled     |
|CLM361656|PRV52066   |BENE50242 |NULL         |NULL        |200.0     |200.75909614900849|0          |H

# ❗ Cell 4 — Key checks (PK + FK validation)

In [6]:
# Cell 4 — DQ: PK + FK null checks

# HARD check — Claim_ID must exist
dq_expect(
    claims,
    "pk_not_null",
    "Claim_ID IS NOT NULL",
    "error",
    "claims",
    paths_silver
)

# SOFT check — Provider_ID (warn)
dq_expect(
    claims,
    "provider_fk_not_null",
    "Provider_ID IS NOT NULL",
    "warn",
    "claims",
    paths_silver
)

# SOFT check — Member_Key (warn)
dq_expect(
    claims,
    "member_fk_not_null",
    "Member_Key IS NOT NULL",
    "warn",
    "claims",
    paths_silver
)


✅ DQ PASS [claims] pk_not_null


❌ DQ FAIL [claims] provider_fk_not_null: 39142/558211 (7.012%)


[QUARANTINE] provider_fk_not_null → abfss://silverdata@clientdatastorage.dfs.core.windows.net/_quarantine/claims/provider_fk_not_null (rows=39142)
✅ DQ PASS [claims] member_fk_not_null


# ✅ Cell 5 — FK validation against Silver Members & Providers

In [7]:
# Cell 5 — FK validation

# Normalize member keys
members_clean = members_bz.select(
    F.col("Member_ID").alias("Member_ID"),
    F.col("Policy_ID")
)

# Normalize provider keys
providers_clean = providers_bz.select(
    F.col("Provider_ID").alias("Provider_ID")
)

# Validate Member FK
dq_left_anti_ref(
    claims, 
    members_clean, 
    "Member_Key", 
    "Member_ID",
    "member_fk_valid",
    "warn",
    "claims",
    paths_silver
)

# Validate Provider FK
dq_left_anti_ref(
    claims, 
    providers_clean, 
    "Provider_ID", 
    "Provider_ID",
    "provider_fk_valid",
    "warn",
    "claims",
    paths_silver
)


❌ DQ FAIL [claims] member_fk_valid: 558211/558211 (100.0%) missing in reference


[QUARANTINE] member_fk_valid → abfss://silverdata@clientdatastorage.dfs.core.windows.net/_quarantine/claims/member_fk_valid (rows=558211)


❌ DQ FAIL [claims] provider_fk_valid: 39142/558211 (7.012%) missing in reference


[QUARANTINE] provider_fk_valid → abfss://silverdata@clientdatastorage.dfs.core.windows.net/_quarantine/claims/provider_fk_valid (rows=39142)


# ✅ Cell 6 — Deduplication & Normalisation

In [8]:
# Cell 6 — Deduplication

claims = drop_dupes_keep_latest(
    claims,
    key_cols=["Claim_ID"],
    order_desc_cols=["Date_Reported"]
)

print("Claims after dedup:", claims.count())


Claims after dedup: 558211


# ✅ Cell 7 — Feature engineering + DQ flags

In [9]:
# Cell 7 — Feature engineering + DQ flags

claims = (
    claims
    .withColumn("Days_To_Settle",
                F.datediff("Date_Settled", "Date_Reported"))
    .withColumn("dq_money_valid",
                F.when(
                    (F.col("Payout_GBP") >= 0) &
                    (F.col("Claim_Amount_GBP") >= 0),
                    1
                ).otherwise(0))
    .withColumn("dq_date_valid",
                F.when(
                    (F.col("Date_Reported").isNull()) |
                    (F.col("Date_Settled").isNull()) |
                    (F.col("Date_Settled") >= F.col("Date_Reported")),
                    1
                ).otherwise(0))
)

claims.show(5)


+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+--------------+--------------+-------------+
| Claim_ID|Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|  Claim_Amount_GBP|Fraud_Label|Claim_Type|Claim_Status|Days_To_Settle|dq_money_valid|dq_date_valid|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+--------------+--------------+-------------+
|CLM110014|   PRV56008| BENE15764|         NULL|        NULL|    1900.0|2005.1467506822025|          0|  Hospital|     Settled|          NULL|             1|            1|
|CLM110042|   PRV52985|BENE124390|         NULL|        NULL|    1600.0|1970.0794650729777|          0|  Hospital|     Settled|          NULL|             1|            1|
|CLM110045|   PRV56645|BENE132290|         NULL|        NULL|    2300.0|3085.1655321856606|          0|Outpatient|     Settled|          NUL

# ✅ Cell 8 — Write Claims Silver (Delta + Table)

In [17]:
# Cell 8 — Write Silver Claims

print("Writing Silver Claims to:", SILVER_CLAIMS_PATH)

(claims.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_CLAIMS_PATH))

spark.sql("CREATE DATABASE IF NOT EXISTS bupa_silver")

spark.sql("""
CREATE TABLE IF NOT EXISTS bupa_silver.claims
USING DELTA
LOCATION '{}'
""".format(SILVER_CLAIMS_PATH))

print("Silver claims written + table registered.")


Writing Silver Claims to: abfss://silverdata@clientdatastorage.dfs.core.windows.net/claims


25/11/28 15:15:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Silver claims written + table registered.


# ✅ Cell 9 — Metrics Logging

In [12]:
from utils_silver import write_metric
from pyspark.sql import functions as F

# Cell 9 — Metrics for Claims Silver

write_metric(
    spark,
    "rowcount_claims_silver",
    claims.count(),
    "claims_silver",
    paths_silver
)

write_metric(
    spark,
    "distinct_claim_ids",
    claims.select("Claim_ID").distinct().count(),
    "claims_silver",
    paths_silver
)

metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(truncate=False)


[METRIC] rowcount_claims_silver=558211 ctx=claims_silver


[METRIC] distinct_claim_ids=558211 ctx=claims_silver


+--------------------------+------+---------------+--------------------------+
|metric                    |value |context        |ts                        |
+--------------------------+------+---------------+--------------------------+
|distinct_claim_ids        |558211|claims_silver  |2025-11-28 15:31:16.213598|
|rowcount_claims_silver    |558211|claims_silver  |2025-11-28 15:30:54.237362|
|distinct_member_ids       |381109|members_silver |2025-11-28 14:04:23.229343|
|rowcount_members_silver   |381109|members_silver |2025-11-28 14:04:14.937709|
|distinct_policy_ids_silver|381109|policies_silver|2025-11-28 10:36:21.144919|
|rowcount_policies_silver  |381109|policies_silver|2025-11-28 10:36:03.700764|
|distinct_policy_ids       |381109|policies_silver|2025-11-25 01:31:29.412102|
|rowcount_policies_silver  |381109|policies_silver|2025-11-25 01:31:06.610274|
+--------------------------+------+---------------+--------------------------+



# ✅ Cell 10 — Final output preview

In [14]:
# Cell 10 — Preview Silver Claims

df = spark.read.format("delta").load(SILVER_CLAIMS_PATH)

print("Silver Claims:", df.count())
df.show(20, truncate=False)
df.printSchema()


Silver Claims: 558211


+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+--------------+--------------+-------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Days_To_Settle|dq_money_valid|dq_date_valid|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+--------------+--------------+-------------+
|CLM110031|NULL       |BENE97876 |NULL         |NULL        |2500.0    |2511.1119392437663|0          |Outpatient|Settled     |NULL          |1             |1            |
|CLM110044|PRV54888   |BENE129291|NULL         |NULL        |70.0      |82.82637935520269 |0          |Hospital  |Settled     |NULL          |1             |1            |
|CLM110046|PRV53609   |BENE132421|NULL         |NULL        |1800.0    |2607.4488956530145|0          |Hospital  |Settled     |NULL         

# 🔍 A/B/C/D — Validation Cells

# A) Bronze vs Silver row comparison

In [15]:
bronze = claims_bz
silver = spark.read.format("delta").load(SILVER_CLAIMS_PATH)

print("BRONZE rows:", bronze.count())
print("SILVER rows:", silver.count())


BRONZE rows: 558211
SILVER rows: 558211


# B) Column comparison

In [16]:
bronze_cols = set(bronze.columns)
silver_cols = set(silver.columns)

print("New in Silver:", silver_cols - bronze_cols)
print("Dropped:", bronze_cols - silver_cols)
print("Common:", sorted(bronze_cols & silver_cols))


New in Silver: {'dq_money_valid', 'dq_date_valid', 'Days_To_Settle'}
Dropped: set()
Common: ['Claim_Amount_GBP', 'Claim_ID', 'Claim_Status', 'Claim_Type', 'Date_Reported', 'Date_Settled', 'Fraud_Label', 'Member_Key', 'Payout_GBP', 'Provider_ID']


# C) Type and value changes

In [17]:
common = sorted(bronze_cols & silver_cols)

for col in common:
    b_type = dict(bronze.dtypes)[col]
    s_type = dict(silver.dtypes)[col]
    print(f"\nColumn: {col}")
    print(" - Bronze:", b_type)
    print(" - Silver:", s_type)

    diffs = silver.alias("s").join(
        bronze.alias("b"),
        on="Claim_ID",
        how="inner"
    ).filter(F.col(f"s.{col}") != F.col(f"b.{col}")).count()

    print("Differences:", diffs)



Column: Claim_Amount_GBP
 - Bronze: double
 - Silver: double


Differences: 0

Column: Claim_ID
 - Bronze: string
 - Silver: string
Differences: 0

Column: Claim_Status
 - Bronze: string
 - Silver: string


Differences: 0

Column: Claim_Type
 - Bronze: string
 - Silver: string


Differences: 0

Column: Date_Reported
 - Bronze: date
 - Silver: date
Differences: 0

Column: Date_Settled
 - Bronze: date
 - Silver: date
Differences: 0

Column: Fraud_Label
 - Bronze: int
 - Silver: int


Differences: 0

Column: Member_Key
 - Bronze: string
 - Silver: string


Differences: 0

Column: Payout_GBP
 - Bronze: double
 - Silver: double


Differences: 0

Column: Provider_ID
 - Bronze: string
 - Silver: string


Differences: 0


# D) Explain new Silver features

In [18]:
silver.select(
    "Claim_ID",
    "Days_To_Settle",
    "dq_money_valid",
    "dq_date_valid"
).show(10)


+---------+--------------+--------------+-------------+
| Claim_ID|Days_To_Settle|dq_money_valid|dq_date_valid|
+---------+--------------+--------------+-------------+
|CLM110031|          NULL|             1|            1|
|CLM110044|          NULL|             1|            1|
|CLM110046|          NULL|             1|            1|
|CLM110050|          NULL|             1|            1|
|CLM110055|          NULL|             1|            1|
|CLM110066|          NULL|             1|            1|
|CLM110073|          NULL|             1|            1|
|CLM110085|          NULL|             1|            1|
|CLM110089|          NULL|             1|            1|
|CLM110092|          NULL|             1|            1|
+---------+--------------+--------------+-------------+
only showing top 10 rows

